# Module 4: Attack Module

This notebook is the final split-out stage of Module 4 and the only split notebook that runs federated learning. It loads the centralized MobileNetV3 target checkpoint and MobileNetV2 surrogate checkpoint, evaluates surrogate and transfer attacks, then runs clean-vs-attacked FedAvg initialized from the V3 checkpoint.

Run this notebook from the `4_Adversarial_FL/` directory after `train_v3.ipynb` and `train_surrogate.ipynb`.


## Stage Goal

Use the saved target and surrogate artifacts to separate three questions:

1. How vulnerable is the surrogate to random noise, FGSM, and PGD?
2. Do surrogate-crafted examples transfer to the centralized MobileNetV3 target?
3. Starting from the same V3 checkpoint, how does FedAvg behave with honest clients versus malicious clients?

The malicious-fraction sweep remains guarded by `attack_module.run_malicious_fraction_sweep` in `attack_module_config.yaml`.


## 1. Notebook Setup

Load the FL server, attack functions, model classes, and evaluation helpers. Unlike the training notebooks, this notebook intentionally constructs a FedAvg server.


In [ ]:
from copy import deepcopy
from pathlib import Path
import os

import json
import yaml
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from algos import Server
from attacks import ATTACK_FUNCTIONS, AttackFn, pixel_linf_norm
from malicious_client import MaliciousClient
from model import MobileNetV2Transfer, MobileNetV3Transfer
from util_functions import (
    create_data,
    evaluate_fn,
    set_logger,
    set_seed,
    target_label_prediction_rate,
)


## 2. Configuration and Artifact Setup

Load `attack_module_config.yaml`, resolve the device, validate the FedAvg attack settings, and create the artifact directory.


In [ ]:
CONFIG_PATH = Path("attack_module_config.yaml")
if not CONFIG_PATH.exists():
    raise FileNotFoundError("Could not locate attack_module_config.yaml in the working directory")
with CONFIG_PATH.open() as f:
    CONFIG = yaml.safe_load(f)


def _merge_dicts(base: dict | None, override: dict | None) -> dict:
    merged = deepcopy(base or {})
    for key, value in (override or {}).items():
        if isinstance(value, dict) and isinstance(merged.get(key), dict):
            merged[key] = _merge_dicts(merged[key], value)
        else:
            merged[key] = deepcopy(value)
    return merged


def _flatten_surrogate_training_config(config: dict) -> dict:
    cfg = _merge_dicts(config.get("surrogate", {}), config.get("surrogate_training", {}))
    optimizer_cfg = cfg.get("optimizer", {}) if isinstance(cfg.get("optimizer", {}), dict) else {}
    criterion_cfg = cfg.get("criterion", {}) if isinstance(cfg.get("criterion", {}), dict) else {}

    if "lr" in optimizer_cfg:
        cfg["learning_rate"] = optimizer_cfg["lr"]
        cfg["lr"] = optimizer_cfg["lr"]
    if "weight_decay" in optimizer_cfg:
        cfg["weight_decay"] = optimizer_cfg["weight_decay"]
    if "label_smoothing" in criterion_cfg:
        cfg["label_smoothing"] = criterion_cfg["label_smoothing"]
    return cfg


global_config = deepcopy(CONFIG.get("global_config", {}))
data_config = deepcopy(CONFIG.get("data_config", {}))
model_config = deepcopy(CONFIG.get("model_config", {}))
alg_configs = deepcopy(CONFIG.get("algorithms", {}))
attack_defaults = deepcopy(CONFIG.get("attack", {}))
artifact_config = deepcopy(CONFIG.get("artifacts", {}))
surrogate_training_config = _flatten_surrogate_training_config(CONFIG)
attack_module_config = deepcopy(CONFIG.get("attack_module", {}))

DEFAULT_FED_CONFIG = deepcopy(alg_configs.get("FedAvg", {}).get("fed_config", {}))
RUN_SURROGATE_ATTACKS = bool(attack_module_config.get("run_surrogate_attacks", True))
RUN_TRANSFER_ATTACKS = bool(attack_module_config.get("run_transfer_attacks", True))
RUN_FEDERATED_POISONING = bool(attack_module_config.get("run_federated_poisoning", True))
RUN_MALICIOUS_FRACTION_SWEEP = bool(attack_module_config.get("run_malicious_fraction_sweep", False))
MALICIOUS_FRACTION_GRID = [float(frac) for frac in attack_module_config.get("malicious_fraction_grid", [0.0, 0.05, 0.1, 0.2])]
set_seed(global_config.get("seed", 42))


def get_device(preferred: str | None = None) -> torch.device:
    choice = preferred if preferred is not None else global_config.get("device")
    if isinstance(choice, str):
        if choice.startswith("cuda") and not torch.cuda.is_available():
            choice = "cpu"
        return torch.device(choice)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = get_device()
global_config["device"] = str(DEVICE)

ARTIFACT_DIR = Path(artifact_config.get("dir", "artifacts"))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def artifact_path(key: str, default_filename: str) -> Path:
    return ARTIFACT_DIR / artifact_config.get(key, default_filename)


def validate_attack_config() -> None:
    issues = []
    fed_cfg = DEFAULT_FED_CONFIG
    attack_cfg = attack_defaults.get("attack", {})
    num_rounds = int(fed_cfg.get("num_rounds", 0))
    start_round = int(attack_defaults.get("start_round", 0))
    malicious_fraction = float(attack_defaults.get("malicious_fraction", 0.0))
    num_classes = int(model_config.get("kwargs", {}).get("num_classes", 10))
    target_label = attack_cfg.get("target_label")
    epsilon = float(attack_cfg.get("epsilon", 0.0))
    step_size = float(attack_cfg.get("step_size", 0.0))
    iters = int(attack_cfg.get("iters", 0))

    if "FedAvg" not in alg_configs:
        issues.append("algorithms.FedAvg is required because attack_module.ipynb is the FedAvg notebook.")
    if num_rounds <= start_round:
        issues.append(f"FedAvg.num_rounds ({num_rounds}) must be greater than attack.start_round ({start_round}).")
    if RUN_FEDERATED_POISONING and malicious_fraction <= 0:
        issues.append("attack.malicious_fraction must be > 0 for attacked FedAvg recipes.")
    if target_label is None or not 0 <= int(target_label) < num_classes:
        issues.append(f"attack.attack.target_label must be in [0, {num_classes - 1}].")
    if epsilon <= 0:
        issues.append("attack.attack.epsilon must be positive.")
    if step_size <= 0:
        issues.append("attack.attack.step_size must be positive.")
    if iters <= 0:
        issues.append("attack.attack.iters must be positive for PGD.")
    if step_size > epsilon:
        issues.append("attack.attack.step_size should not exceed epsilon for PGD.")
    for frac in MALICIOUS_FRACTION_GRID:
        if not 0.0 <= frac <= 1.0:
            issues.append(f"attack_module.malicious_fraction_grid contains invalid fraction {frac}.")

    if issues:
        raise ValueError("Attack-module config validation failed:\n" + "\n".join(issues))

    print(
        "Attack module config ready: "
        f"device={DEVICE}, FedAvg rounds={num_rounds}, attack starts at round {start_round}, "
        f"malicious_fraction={malicious_fraction}, epsilon={epsilon:.5f}, "
        f"fraction_sweep={RUN_MALICIOUS_FRACTION_SWEEP}"
    )


validate_attack_config()

config_snapshot = deepcopy(CONFIG)
config_snapshot.setdefault("global_config", {})["resolved_device"] = str(DEVICE)
config_path = artifact_path("config_snapshot", "module4_config_used.json")
with config_path.open("w") as f:
    json.dump(config_snapshot, f, indent=2)

print("Loaded config from", CONFIG_PATH.resolve())
print(f"Saved config snapshot to {config_path.resolve()}")


## 3. Load Target, Surrogate, and Evaluation Data

Load the centralized V3 target checkpoint, the centralized V2 surrogate checkpoint, and a shared evaluation loader. The FedAvg runs later start from `target_state`.


In [ ]:
def _resolve_existing_artifact(key: str, default_filename: str, legacy_filenames: list[str] | None = None) -> Path:
    path = artifact_path(key, default_filename)
    if path.exists():
        return path
    for filename in legacy_filenames or []:
        legacy_path = ARTIFACT_DIR / filename
        if legacy_path.exists():
            print(f"Using legacy artifact name for {key}: {legacy_path}")
            return legacy_path
    return path


def _load_state_dict(path: Path) -> dict[str, torch.Tensor]:
    state = torch.load(path, map_location="cpu")
    if isinstance(state, dict) and "model_state" in state:
        return state["model_state"]
    if isinstance(state, dict) and "state_dict" in state:
        return state["state_dict"]
    return state


def _checkpoint_model_config() -> dict:
    cfg = deepcopy(model_config)
    cfg.setdefault("kwargs", {})["pretrained"] = False
    return cfg


def build_target_model_from_checkpoint() -> torch.nn.Module:
    kwargs = deepcopy(model_config.get("kwargs", {}))
    kwargs["pretrained"] = False
    target = MobileNetV3Transfer(**kwargs).to(DEVICE)
    target.load_state_dict(target_state)
    target.eval()
    return target


target_checkpoint_path = _resolve_existing_artifact(
    "target_checkpoint",
    "module4_v3_target.pt",
    legacy_filenames=["module4_fedavg_target.pt"],
)
target_summary_path = artifact_path("target_metrics", "module4_target_training.json")
surrogate_checkpoint_path = artifact_path("surrogate_checkpoint", "module4_surrogate.pt")
surrogate_summary_path = artifact_path("surrogate_metrics", "module4_surrogate.json")

missing = [
    path
    for path in [target_checkpoint_path, surrogate_checkpoint_path, surrogate_summary_path]
    if not path.exists()
]
if missing:
    missing_text = "\n".join(str(path) for path in missing)
    raise FileNotFoundError(
        "Run train_v3.ipynb and train_surrogate.ipynb before this attack notebook. "
        f"Missing required artifact(s):\n{missing_text}"
    )

with surrogate_summary_path.open("r") as f:
    surrogate_summary = json.load(f)
if target_summary_path.exists():
    with target_summary_path.open("r") as f:
        target_summary = json.load(f)
else:
    target_summary = {}
    print(f"Target summary not found at {target_summary_path}; continuing with checkpoint-only target state.")

target_state = _load_state_dict(target_checkpoint_path)
num_classes = int(model_config.get("kwargs", {}).get("num_classes", 10))
surrogate_model = MobileNetV2Transfer(pretrained=False, num_classes=num_classes).to(DEVICE)
surrogate_state = _load_state_dict(surrogate_checkpoint_path)
surrogate_model.load_state_dict(surrogate_state)
surrogate_model.eval()

target_model = build_target_model_from_checkpoint()
attack_train_dataset, attack_test_dataset = create_data(
    data_config["dataset_path"],
    data_config["dataset_name"],
)
attack_eval_loader = DataLoader(
    attack_test_dataset,
    batch_size=int(attack_module_config.get("eval_batch_size", 512)),
    shuffle=False,
    drop_last=False,
    num_workers=int(attack_module_config.get("num_workers", 0)),
    pin_memory=bool(attack_module_config.get("pin_memory", DEVICE.type == "cuda")),
)
attack_criterion = nn.CrossEntropyLoss().to(DEVICE)
target_eval_loss, target_eval_acc = evaluate_fn(attack_eval_loader, target_model, attack_criterion, DEVICE)
baseline_results: dict[str, dict] = {}

print(f"Loaded MobileNetV3 target checkpoint from {target_checkpoint_path.resolve()}")
print(f"Loaded MobileNetV2 surrogate checkpoint from {surrogate_checkpoint_path.resolve()}")
print(f"Target checkpoint evaluation: loss={target_eval_loss:.4f}, accuracy={target_eval_acc:.2f}%")
print(f"Surrogate recorded test accuracy: {float(surrogate_summary.get('test_accuracy', 0.0)):.2f}%")


## 4. FedAvg Helpers

These helpers run clean or attacked FedAvg from the centralized V3 checkpoint. The clean and attacked runs share the same initialization, client split, rounds, and summary fields.


In [ ]:
def _clear_cuda_cache() -> None:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _to_builtin(value):
    if isinstance(value, torch.Tensor):
        value = value.detach().cpu().item()
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, (int, float, str, bool)) or value is None:
        return value
    return float(value)


def _set_server_initial_state(server, state: dict[str, torch.Tensor]) -> None:
    server.global_model.load_state_dict(state)
    server.global_model.to(server.device)
    server.x = server.global_model


def train_fedavg_server(attack_cfg: dict | None = None):
    alg_conf = alg_configs["FedAvg"]
    fed_cfg = deepcopy(alg_conf["fed_config"])
    fed_cfg["algorithm"] = "FedAvg"
    optim_cfg = deepcopy(alg_conf.get("optim_config", {}))
    attack_cfg = deepcopy(attack_cfg or {"malicious_fraction": 0.0})

    logs_dir = os.path.join("Logs", "FedAvg", str(data_config.get("non_iid_per", 0.0)))
    os.makedirs(logs_dir, exist_ok=True)
    set_logger(os.path.join(logs_dir, "log.txt"))

    server = Server(
        _checkpoint_model_config(),
        global_config,
        data_config,
        fed_cfg,
        optim_cfg,
        attack_cfg,
    )
    server.setup()
    _set_server_initial_state(server, target_state)
    print(f"Initialized FedAvg global model from {target_checkpoint_path.name}")
    server.train()
    return server


def _target_label_from_attack_config(attack_cfg: dict | None) -> int | None:
    attack_params = (attack_cfg or {}).get("attack", {})
    target_label = attack_params.get("target_label")
    return int(target_label) if target_label is not None else None


def summarise_server(server, target_label: int | None = None) -> dict:
    model = getattr(server, "global_model", getattr(server, "x", None))
    loss, acc = evaluate_fn(server.test_loader, model, server.criterion, server.device)
    raw_history = server.results if hasattr(server, "results") else {}
    history = {}
    for key, values in raw_history.items():
        if isinstance(values, (list, tuple)):
            history[key] = [_to_builtin(value) for value in values]

    summary = {
        "initial_checkpoint": str(target_checkpoint_path),
        "final_loss": float(loss),
        "final_accuracy": float(acc),
        "history": history,
        "malicious_client_ids": list(getattr(server, "malicious_client_ids", [])),
    }
    if target_label is not None:
        summary["global_target_label"] = int(target_label)
        summary["global_target_label_asr"] = float(
            target_label_prediction_rate(
                server.test_loader,
                model,
                target_label,
                server.device,
                exclude_true_target_label=True,
            )
        )

    for metric in (
        "surrogate_poison_success_rate",
        "poisoned_examples",
        "candidate_examples",
        "sampled_malicious_clients",
    ):
        values = history.get(metric, [])
        if values:
            summary[f"final_{metric}"] = _to_builtin(values[-1])
    return summary


def run_one_fedavg(attack_cfg: dict | None = None) -> dict:
    server = train_fedavg_server(attack_cfg=attack_cfg)
    summary = summarise_server(
        server,
        target_label=_target_label_from_attack_config(attack_cfg),
    )
    del server
    _clear_cuda_cache()
    return summary


## 5. Attack Configuration

Extract shared attack settings and define the clean, random-noise, FGSM, and PGD recipes.


In [ ]:
SURROGATE_CFG = surrogate_training_config
SURROGATE_CLIENT_ID = int(SURROGATE_CFG.get("client_id", 0))
SURROGATE_BATCH_SIZE = int(SURROGATE_CFG.get("batch_size", DEFAULT_FED_CONFIG.get("batch_size", 64)))
ATTACK_RAW = attack_defaults
ATTACK_BASE = ATTACK_RAW.get("attack", {})
SURROGATE_BASE = ATTACK_RAW.get("surrogate", {})
ATTACK_SEED = ATTACK_RAW.get("seed", global_config.get("seed", 42))
BASE_MALICIOUS_FRACTION = ATTACK_RAW.get("malicious_fraction", 0.0)
ATTACK_EPSILON = float(ATTACK_BASE.get("epsilon", 0.03137255))


def _extract_attack_params(overrides: dict | None = None) -> dict:
    params = {
        "type": ATTACK_BASE.get("type", "pgd"),
        "poison_rate": ATTACK_BASE.get("poison_rate", 0.0),
        "target_label": ATTACK_BASE.get("target_label", 0),
        "epsilon": ATTACK_EPSILON,
        "step_size": ATTACK_BASE.get("step_size", 0.00784314),
        "iters": ATTACK_BASE.get("iters", 10),
        "criterion": ATTACK_BASE.get("criterion", "torch.nn.CrossEntropyLoss"),
    }
    schedule = ATTACK_BASE.get("poison_rate_schedule")
    if schedule:
        params["poison_rate_schedule"] = schedule
    if overrides:
        params.update(overrides)
    return params


def _extract_surrogate_params(overrides: dict | None = None) -> dict:
    params = {
        "pretrained": SURROGATE_CFG.get("pretrained", SURROGATE_BASE.get("pretrained", True)),
        "finetune_epochs": SURROGATE_CFG.get("num_epochs", SURROGATE_BASE.get("finetune_epochs", 0)),
        "lr": SURROGATE_CFG.get("learning_rate", SURROGATE_BASE.get("lr", 1e-3)),
        "weight_decay": SURROGATE_CFG.get("weight_decay", SURROGATE_BASE.get("weight_decay", 0.0)),
        "batch_size": SURROGATE_BATCH_SIZE,
        "client_id": SURROGATE_CLIENT_ID,
        "pool_size": SURROGATE_CFG.get("pool_size", SURROGATE_BASE.get("pool_size", 1)),
        "freeze_backbone": SURROGATE_CFG.get("freeze_backbone", SURROGATE_BASE.get("freeze_backbone", False)),
        "augment": SURROGATE_CFG.get("augment", SURROGATE_BASE.get("augment", False)),
        "early_stop_patience": SURROGATE_CFG.get("early_stop_patience", SURROGATE_BASE.get("early_stop_patience", 0)),
        "num_classes": SURROGATE_CFG.get("num_classes", SURROGATE_BASE.get("num_classes", num_classes)),
    }
    if overrides:
        params.update(overrides)
    return params


def build_attack_config(*, attack_overrides: dict | None = None, surrogate_overrides: dict | None = None, malicious_fraction: float | None = None, seed: int | None = None) -> dict:
    cfg = {
        "seed": seed if seed is not None else ATTACK_SEED,
        "malicious_fraction": BASE_MALICIOUS_FRACTION if malicious_fraction is None else malicious_fraction,
        "attack": _extract_attack_params(attack_overrides),
        "surrogate": _extract_surrogate_params(surrogate_overrides),
    }
    for key in ("start_round",):
        if key in ATTACK_RAW:
            cfg[key] = ATTACK_RAW[key]
    return cfg


ATTACK_RECIPES = {
    "clean": build_attack_config(malicious_fraction=0.0, attack_overrides={"poison_rate": 0.0}),
    "pgd_default": build_attack_config(),
    "fgsm_default": build_attack_config(attack_overrides={"type": "fgsm", "step_size": ATTACK_EPSILON, "iters": 1}),
    "random_noise": build_attack_config(attack_overrides={"type": "random_noise", "step_size": ATTACK_EPSILON}),
}

print("Attack budgets are interpreted in pixel space, then normalized for MobileNet evaluation.")


## 6. Malicious Client and Attack Helpers

These helpers craft adversarial batches with the loaded surrogate checkpoint and run FedAvg attack recipes from the loaded V3 checkpoint.


In [ ]:
def select_attack_fn(name: str) -> AttackFn:
    key = name.lower()
    if key not in ATTACK_FUNCTIONS:
        raise KeyError(f"Unknown attack function '{name}'. Available: {sorted(ATTACK_FUNCTIONS)}")
    return ATTACK_FUNCTIONS[key]


def _attach_attack_callable(cfg: dict) -> dict:
    cfg_copy = deepcopy(cfg)
    attack_params = cfg_copy.setdefault("attack", {})
    attack_type = attack_params.get("type", "pgd")
    if "callable" not in attack_params:
        attack_params["callable"] = select_attack_fn(attack_type)
    return cfg_copy


def attack_label(recipe_name: str) -> str:
    return {
        "random_noise": "Random",
        "fgsm_default": "FGSM",
        "pgd_default": "PGD",
        "clean": "Clean",
    }.get(recipe_name, recipe_name)


def make_malicious_client(
    attack_config: dict,
    *,
    local_loader=None,
    surrogate=None,
    num_epochs: int = 0,
    lr: float | None = None,
):
    if local_loader is None:
        local_loader = attack_eval_loader
    if surrogate is None:
        surrogate = surrogate_model

    client = MaliciousClient(
        client_id=SURROGATE_CLIENT_ID,
        local_data=local_loader,
        device=DEVICE,
        num_epochs=num_epochs,
        criterion=nn.CrossEntropyLoss().to(DEVICE),
        lr=lr if lr is not None else DEFAULT_FED_CONFIG.get("local_stepsize", 0.003),
        attack_config=attack_config,
    )
    client.surrogate = surrogate.to(DEVICE)
    client.surrogate.eval()
    return client


def craft_adversarial_batch(client: MaliciousClient, inputs: torch.Tensor, labels: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    attack_params = client.attack_params
    attack_type = attack_params.get("type", "pgd").lower()
    target_label = int(attack_params.get("target_label", 0))
    target_labels = torch.full_like(labels, target_label)
    targeted = bool(attack_params.get("targeted", attack_params.get("target_label") is not None))

    if attack_type in {"pgd", "fgsm"}:
        attack_fn = select_attack_fn(attack_type)
        kwargs = {
            "model": client.surrogate,
            "criterion": client.attack_criterion,
            "images": inputs,
            "labels": target_labels,
            "step_size": float(attack_params.get("step_size", ATTACK_EPSILON)),
            "targeted": targeted,
            "normalized": True,
        }
        if attack_type == "pgd":
            kwargs["eps"] = float(attack_params.get("epsilon", ATTACK_EPSILON))
            kwargs["iters"] = int(attack_params.get("iters", 10))
        adv = attack_fn(**kwargs)
    else:
        attack_fn = select_attack_fn("random_noise")
        adv = attack_fn(
            inputs,
            step_size=float(attack_params.get("step_size", attack_params.get("epsilon", ATTACK_EPSILON))),
            normalized=True,
        )

    return adv, target_labels


def _next_eval_batch(batch_size: int):
    inputs, labels = next(iter(attack_eval_loader))
    return inputs[:batch_size].to(DEVICE).float(), labels[:batch_size].to(DEVICE).long()


def _attack_metrics(model: torch.nn.Module, inputs: torch.Tensor, labels: torch.Tensor, adv_inputs: torch.Tensor, adv_labels: torch.Tensor) -> dict:
    model.eval()
    with torch.no_grad():
        clean_logits = model(inputs)
        clean_preds = clean_logits.argmax(dim=1)
        adv_logits = model(adv_inputs)
        adv_preds = adv_logits.argmax(dim=1)
    clean_acc = (clean_preds == labels).float().mean().item() * 100.0
    robust_acc = (adv_preds == labels).float().mean().item() * 100.0
    target_label_success_rate = (adv_preds == adv_labels).float().mean().item() * 100.0
    misclassification_rate = (adv_preds != labels).float().mean().item() * 100.0
    linf = pixel_linf_norm(inputs, adv_inputs, normalized=True)
    return {
        "clean_accuracy": clean_acc,
        "robust_accuracy": robust_acc,
        "target_label_success_rate": target_label_success_rate,
        "misclassification_rate": misclassification_rate,
        "mean_linf": linf.mean().item(),
        "max_linf": linf.max().item(),
    }


def evaluate_surrogate_attack(recipe_name: str, *, batch_size: int = 32) -> dict:
    attack_cfg = _attach_attack_callable(ATTACK_RECIPES[recipe_name])
    client = make_malicious_client(attack_cfg, surrogate=surrogate_model, num_epochs=0)
    inputs, labels = _next_eval_batch(batch_size)
    adv_inputs, adv_labels = craft_adversarial_batch(client, inputs, labels)
    metrics = _attack_metrics(surrogate_model, inputs, labels, adv_inputs, adv_labels)
    metrics.update({
        "recipe": recipe_name,
        "attack": attack_label(recipe_name),
        "epsilon": float(attack_cfg["attack"].get("epsilon", ATTACK_EPSILON)),
        "surrogate_target_label_success_rate": metrics.pop("target_label_success_rate"),
        "surrogate_misclassification_rate": metrics.pop("misclassification_rate"),
    })
    return metrics


def evaluate_transfer_attack(recipe_name: str, target_model: torch.nn.Module, *, batch_size: int = 32) -> dict:
    attack_cfg = _attach_attack_callable(ATTACK_RECIPES[recipe_name])
    client = make_malicious_client(attack_cfg, surrogate=surrogate_model, num_epochs=0)
    inputs, labels = _next_eval_batch(batch_size)
    adv_inputs, adv_labels = craft_adversarial_batch(client, inputs, labels)
    metrics = _attack_metrics(target_model, inputs, labels, adv_inputs, adv_labels)
    metrics.update({
        "recipe": recipe_name,
        "attack": attack_label(recipe_name),
        "epsilon": float(attack_cfg["attack"].get("epsilon", ATTACK_EPSILON)),
        "target_clean_accuracy": metrics.pop("clean_accuracy"),
        "target_robust_accuracy": metrics.pop("robust_accuracy"),
        "target_model_target_label_success_rate": metrics.pop("target_label_success_rate"),
        "surrogate_to_target_transfer_success_rate": metrics.pop("misclassification_rate"),
    })
    return metrics


def run_attack_recipe_on_server(recipe_name: str, malicious_fraction: float | None = None) -> dict:
    attack_cfg = _attach_attack_callable(ATTACK_RECIPES[recipe_name])
    if malicious_fraction is not None:
        attack_cfg = deepcopy(attack_cfg)
        attack_cfg["malicious_fraction"] = malicious_fraction
    summary = run_one_fedavg(attack_cfg=attack_cfg)
    summary.update({"recipe": recipe_name, "algorithm": "FedAvg"})
    return summary


def sweep_attacks_on_server(recipes: list[str] | None = None, malicious_fraction: float | None = None) -> dict:
    global baseline_results
    recipes = recipes or list(ATTACK_RECIPES)
    results = {}
    for recipe in recipes:
        if recipe == "clean":
            if "FedAvg" not in baseline_results:
                baseline_results["FedAvg"] = run_attack_recipe_on_server("clean", malicious_fraction=0.0)
            results[recipe] = deepcopy(baseline_results["FedAvg"])
            continue
        results[recipe] = run_attack_recipe_on_server(
            recipe,
            malicious_fraction=malicious_fraction,
        )
    return results


## 7. Random Noise, FGSM, and PGD on the Surrogate

Run the three attack recipes against MobileNetV2 only. This checks whether the loaded surrogate is vulnerable under the configured pixel budget.


In [ ]:
SURROGATE_ATTACK_RECIPES = ["random_noise", "fgsm_default", "pgd_default"]

if RUN_SURROGATE_ATTACKS:
    surrogate_attack_results = {
        recipe: evaluate_surrogate_attack(recipe)
        for recipe in SURROGATE_ATTACK_RECIPES
    }
else:
    surrogate_attack_results = {}
    print("Skipping surrogate attacks; set attack_module.run_surrogate_attacks=true in attack_module_config.yaml to run them.")

surrogate_attack_results


### Save and Plot Surrogate Attack Metrics

Save the surrogate-only attack summary and plot clean accuracy, robust accuracy, and target-label success.


In [ ]:
sur_attack_path = artifact_path("surrogate_attack_metrics", "module4_surrogate_attacks.json")
with sur_attack_path.open("w") as f:
    json.dump(surrogate_attack_results, f, indent=2)
print(f"Saved surrogate attack metrics to {sur_attack_path.resolve()}")

import pandas as pd


def plot_surrogate_attack_results(results: dict[str, dict]) -> None:
    if not results:
        print("No surrogate attack results to plot for this run mode.")
        return None
    df = pd.DataFrame(results).T
    labels = [row["attack"] for _, row in df.iterrows()]
    x = np.arange(len(df))
    width = 0.25
    plt.figure(figsize=(7, 4))
    plt.bar(x - width, df["clean_accuracy"], width=width, label="Clean acc")
    plt.bar(x, df["robust_accuracy"], width=width, label="Robust acc")
    plt.bar(x + width, df["surrogate_target_label_success_rate"], width=width, label="Target-label success")
    plt.xticks(x, labels)
    plt.ylabel("Percentage")
    plt.title("Surrogate attack sanity check")
    plt.legend()
    plt.tight_layout()
    out_path = artifact_path("surrogate_attack_plot", "surrogate_attack_success_by_attack.png")
    plt.savefig(out_path, dpi=150)
    print(f"Saved {out_path.resolve()}")
    return df[["attack", "epsilon", "clean_accuracy", "robust_accuracy", "surrogate_target_label_success_rate", "mean_linf", "max_linf"]]


plot_surrogate_attack_results(surrogate_attack_results)


## 8. Surrogate-to-Target Transfer

Craft adversarial examples with MobileNetV2 gradients, then evaluate those same tensors on the centralized MobileNetV3 target checkpoint.


In [ ]:
TRANSFER_ATTACK_RECIPES = ["random_noise", "fgsm_default", "pgd_default"]

if RUN_TRANSFER_ATTACKS:
    target_model = build_target_model_from_checkpoint()
    transfer_results = {
        recipe: evaluate_transfer_attack(recipe, target_model)
        for recipe in TRANSFER_ATTACK_RECIPES
    }
else:
    transfer_results = {}
    print("Skipping transfer attacks; set attack_module.run_transfer_attacks=true in attack_module_config.yaml to run them.")

transfer_path = artifact_path("transfer_metrics", "module4_transfer_results.json")
with transfer_path.open("w") as f:
    json.dump(transfer_results, f, indent=2)
print(f"Saved transfer metrics to {transfer_path.resolve()}")

if transfer_results:
    transfer_table = pd.DataFrame(transfer_results).T[
        [
            "attack",
            "epsilon",
            "target_clean_accuracy",
            "target_robust_accuracy",
            "target_model_target_label_success_rate",
            "surrogate_to_target_transfer_success_rate",
            "mean_linf",
            "max_linf",
        ]
    ]
    transfer_table
else:
    print("No transfer table for this run mode.")


## 9. Clean vs. Attacked FedAvg

Run clean FedAvg and PGD-poisoned FedAvg from the same centralized V3 checkpoint. This is the only federated training section in the split Module 4 notebooks.

`attack_module_config.yaml` lists the FedAvg settings used by this malicious-client path. FedOpt and SCAFFOLD attack variants would need additional malicious-client support.


In [ ]:
FED_ATTACK_RECIPES = ["clean", "pgd_default"]

if RUN_FEDERATED_POISONING:
    federated_attack_results = sweep_attacks_on_server(
        recipes=FED_ATTACK_RECIPES,
    )
else:
    federated_attack_results = {}
    print("Skipping federated poisoning; set attack_module.run_federated_poisoning=true in attack_module_config.yaml to run it.")

federated_attack_results


### Save FedAvg Metrics

Save clean FedAvg baseline metrics separately and save the clean-vs-attacked comparison in `module4_federated_attacks.json`.


In [ ]:
if federated_attack_results.get("clean"):
    baseline_results["FedAvg"] = federated_attack_results["clean"]
    baseline_path = artifact_path("baseline_metrics", "module4_federated_baseline.json")
    with baseline_path.open("w") as f:
        json.dump(baseline_results, f, indent=2)
    print(f"Saved clean FedAvg baseline metrics to {baseline_path.resolve()}")

fed_attack_path = artifact_path("federated_attack_metrics", "module4_federated_attacks.json")
with fed_attack_path.open("w") as f:
    json.dump(federated_attack_results, f, indent=2)
print(f"Saved federated attack metrics to {fed_attack_path.resolve()}")


### Federated Attack Sanity Check

Compare final clean and attacked FedAvg accuracy. The ASR metric here is measured on the final MobileNetV3 global model, not on the surrogate.


In [ ]:
clean_acc = federated_attack_results.get("clean", {}).get("final_accuracy")
attack_summary = federated_attack_results.get("pgd_default", {})
attack_acc = attack_summary.get("final_accuracy")
global_asr = attack_summary.get("global_target_label_asr")
if clean_acc is not None and attack_acc is not None:
    delta = clean_acc - attack_acc
    message = f"FedAvg clean accuracy: {clean_acc:.2f}%  |  attacked: {attack_acc:.2f}%  |  drop: {delta:.2f} pts"
    if global_asr is not None:
        message += f"  |  global target-label ASR: {global_asr:.2f}%"
    print(message)
else:
    print("Run the clean and attacked FedAvg cell before executing this check.")


### Federated Attack Impact Plot

Plot the clean and attacked FedAvg accuracy curves across rounds. Inspect whether the attacked curve diverges after the configured attack start round.


In [ ]:
def plot_federated_attack_results(results: dict[str, dict]) -> None:
    clean = results.get("clean", {})
    attacked = results.get("pgd_default", {})
    if not clean or not attacked:
        print("Run the clean and attacked FedAvg cells before plotting.")
        return

    clean_acc = clean.get("history", {}).get("accuracy", [])
    attack_acc = attacked.get("history", {}).get("accuracy", [])

    plt.figure(figsize=(7, 4))
    if clean_acc:
        plt.plot(range(1, len(clean_acc) + 1), clean_acc, marker="o", label="Clean FedAvg")
    if attack_acc:
        plt.plot(range(1, len(attack_acc) + 1), attack_acc, marker="o", label="Attacked FedAvg")
    start_round = int(attack_defaults.get("start_round", 0))
    if start_round:
        plt.axvline(start_round, linestyle="--", color="black", alpha=0.6, label="Attack starts")
    plt.xlabel("Round")
    plt.ylabel("Accuracy (%)")
    plt.title("Clean vs attacked FedAvg from V3 checkpoint")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    out_path = artifact_path("federated_attack_plot", "attack_accuracy.png")
    plt.savefig(out_path, dpi=150)
    print(f"Saved {out_path.resolve()}")


plot_federated_attack_results(federated_attack_results)


## 10. Malicious Fraction Sweep

Optionally vary the fraction of malicious clients and rerun attacked FedAvg. The clean FedAvg baseline is reused because it does not depend on malicious fraction.


In [ ]:
malicious_grid = MALICIOUS_FRACTION_GRID
fraction_sweep_results = {}
comparison_rows = []
if not RUN_MALICIOUS_FRACTION_SWEEP:
    print("Skipping malicious-fraction sweep; set attack_module.run_malicious_fraction_sweep=true in attack_module_config.yaml to run it.")
else:
    if "FedAvg" not in baseline_results:
        baseline_results["FedAvg"] = run_attack_recipe_on_server("clean", malicious_fraction=0.0)
    for frac in malicious_grid:
        fraction_sweep_results[frac] = sweep_attacks_on_server(
            recipes=["clean", "pgd_default"],
            malicious_fraction=frac,
        )

    for frac, results in fraction_sweep_results.items():
        clean_summary = results.get("clean", {})
        attack_summary = results.get("pgd_default", {})
        clean_acc = clean_summary.get("final_accuracy")
        attack_acc = attack_summary.get("final_accuracy")
        comparison_rows.append({
            "malicious_fraction": frac,
            "final_clean_accuracy": clean_acc,
            "final_attacked_accuracy": attack_acc,
            "accuracy_drop": (clean_acc - attack_acc) if clean_acc is not None and attack_acc is not None else None,
            "global_target_label_asr": attack_summary.get("global_target_label_asr", 0.0),
            "surrogate_poison_success_rate": attack_summary.get("final_surrogate_poison_success_rate", 0.0),
            "poisoned_examples": attack_summary.get("final_poisoned_examples", 0),
        })

comparison_rows


### Save and Plot the Malicious-Fraction Sweep

When the optional sweep is enabled, save the sweep table and plot final attacked accuracy, global target-label ASR, and surrogate poison success.


In [ ]:
fraction_path = artifact_path("fraction_sweep_metrics", "module4_fraction_sweep.json")
if comparison_rows:
    with fraction_path.open("w") as f:
        json.dump(comparison_rows, f, indent=2)
    print(f"Saved sweep results to {fraction_path.resolve()}")
else:
    print("No malicious-fraction sweep rows to save for this run mode.")


def plot_fraction_sweep(rows: list[dict] | None) -> None:
    if not rows:
        print("No fraction sweep results available to plot.")
        return
    fractions = [row["malicious_fraction"] for row in rows]
    accuracies = [row.get("final_attacked_accuracy") for row in rows]
    global_target_label_asr = [row.get("global_target_label_asr", 0.0) for row in rows]
    surrogate_poison_success = [row.get("surrogate_poison_success_rate", 0.0) for row in rows]

    plt.figure(figsize=(7, 4))
    plt.plot(fractions, accuracies, marker="o", label="Final attacked accuracy")
    plt.plot(fractions, global_target_label_asr, marker="s", label="Global target-label ASR")
    plt.plot(fractions, surrogate_poison_success, marker="^", label="Surrogate poison success")
    plt.xlabel("Malicious fraction")
    plt.ylabel("Percentage")
    plt.title("Impact of malicious fraction on FedAvg")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    out_path = artifact_path("fraction_sweep_plot", "malicious_fraction_sweep.png")
    plt.savefig(out_path, dpi=150)
    print(f"Saved {out_path.resolve()}")


plot_fraction_sweep(comparison_rows)


## Final Interpretation and Transition to Module 5

Use the generated artifacts to write a short interpretation of the attack pipeline. A complete answer should distinguish corruption robustness, gradient-based adversarial examples, black-box transfer, and malicious-client poisoning.

Checkpoint questions:

1. Did random noise hurt the surrogate?
2. Did FGSM/PGD hurt more than random noise?
3. Did surrogate-crafted attacks transfer to the centralized MobileNetV3 target?
4. Starting from the same V3 checkpoint, did malicious clients reduce global FedAvg accuracy?
5. Did surrogate poison success increase with malicious fraction?
6. Did final global-model target-label ASR move with malicious fraction?
7. Why is surrogate poison success not the same thing as final global-model target-label ASR?
8. Why does this motivate robust aggregation?

**Transition to Module 5:** FedAvg averages malicious and honest updates together, so even a small malicious fraction can influence the global model. Module 5 asks whether robust aggregation, clipping, median, trimmed mean, Krum, Multi-Krum, and related defenses can preserve clean accuracy while reducing attacked accuracy loss and global target-label ASR.


## Artifact Guide

After a default run, inspect these files in `artifacts/` to answer the main Module 4 questions. The malicious-fraction sweep artifacts are written only when the optional sweep is enabled.

| Artifact | What it answers |
| --- | --- |
| `module4_config_used.json` | Which config, device, attack budget, target label, and FedAvg settings produced this run? |
| `module4_target_training.json` | How well did the centralized MobileNetV3 target train before attacks? |
| `module4_v3_target.pt` | Which V3 checkpoint initialized transfer evaluation and clean-vs-attacked FedAvg? |
| `module4_surrogate.json` | Did the centralized MobileNetV2 surrogate train well enough to be a credible attacker model? |
| `module4_surrogate.pt` | Which surrogate checkpoint crafted the FGSM and PGD examples? |
| `module4_surrogate_attacks.json` | On the surrogate, how do random noise, FGSM, and PGD compare for clean accuracy, robust accuracy, and target-label success? |
| `surrogate_attack_success_by_attack.png` | Which attack most clearly separates adversarial behavior from random corruption on the surrogate? |
| `module4_transfer_results.json` | Did MobileNetV2-crafted examples transfer to the centralized MobileNetV3 target, and did they cause target-label predictions? |
| `module4_federated_baseline.json` | How did clean FedAvg behave when initialized from the V3 checkpoint? |
| `module4_federated_attacks.json` | Under the default malicious fraction, how did attacked FedAvg change global accuracy, poisoned-example counts, surrogate poison success, and final `global_target_label_asr`? |
| `attack_accuracy.png` | When the attack started, did attacked FedAvg diverge from clean FedAvg? |
| `module4_fraction_sweep.json` | Optional: as malicious-client fraction changes, how do final attacked accuracy, `global_target_label_asr`, poisoned examples, and surrogate poison success change? |
| `malicious_fraction_sweep.png` | Optional: what visual tradeoff appears between attacked accuracy, global target-label ASR, and surrogate poison success as more clients become malicious? |
